# 01 · Embeddings of the corpus

This notebook turns the corpus of early modern Catholic texts
([catholic-pastoral-texts](https://github.com/vegafabianr/catholic-pastoral-texts))
into **embeddings**: lists of numbers that place every passage in a "semantic space",
where passages that talk about similar things end up close to each other.

The method follows Alammar & Grootendorst, *Hands-On Large Language Models* (2024), ch. 5:
split the texts into passages, embed them, then reduce and cluster (next notebook).

**Run it in Google Colab with a GPU** (*Runtime → Change runtime type → T4 GPU*).
On a GPU it takes about 15–30 minutes; on a laptop CPU it would take many hours.

Output: `embeddings.zip`, with two files:
- `chunks.csv.gz`: one row per passage (which text it comes from, its position, its words).
- `embeddings.npy`: one row of 1,024 numbers per passage, in the same order.

## 1. Settings

Everything you may want to change is here.

In [ ]:
MODEL_NAME = "BAAI/bge-m3"   # multilingual embedding model, strong in Spanish
CHUNK_WORDS = 250            # words per passage (fits well under the model's 512-token sweet spot)
MIN_CHUNK_WORDS = 100        # a final leftover shorter than this is glued to the previous passage
BATCH_SIZE = 64              # passages sent to the GPU at once; lower it if you run out of memory

## 2. Download the corpus and its metadata

We clone the GitHub repository (about 500 MB). `--depth 1` downloads only the latest version.

In [ ]:
!git clone --depth 1 -q https://github.com/vegafabianr/catholic-pastoral-texts corpus

import pandas as pd

# One row per text, with author, religious order, period of death, topic...
texts = pd.read_csv("corpus/metadatos/textos.csv")
print(len(texts), "texts")
texts[["text_id", "author", "author_institution", "author_death_period", "topic"]].head()

## 3. Clean the texts

We use the **modernized** version (current spelling), which language models read better.
The files contain a light XML-like markup that we have to remove:

- `<pb n="12"/>` marks a page break → removed.
- `<head>...</head>` marks a heading → we keep the words, drop the tags.
- `<foreign xml:lang="la">...</foreign>` marks a passage in Latin (mostly biblical quotations)
  → we drop the whole passage, so that Latin does not pull passages together just for being Latin.

Then we remove accent marks (*él* → *el*), but keep the *ñ*, and join words split by a hyphen
at the end of a line.

In [ ]:
import re
import unicodedata

def remove_accents(text):
    """Remove acute, grave, circumflex and diaeresis marks, but keep ñ.

    NFD splits 'á' into 'a' + a separate accent mark; we drop the marks we do not want
    and then NFC glues the remaining pieces back ('n' + tilde becomes 'ñ' again)."""
    unwanted_marks = {"\u0301", "\u0300", "\u0302", "\u0308"}  # ´ ` ^ ¨
    decomposed = unicodedata.normalize("NFD", text)
    pieces = []
    for character in decomposed:
        if character not in unwanted_marks:
            pieces.append(character)
    return unicodedata.normalize("NFC", "".join(pieces))

def clean(text):
    text = re.sub(r"<foreign[^>]*>.*?</foreign>", " ", text, flags=re.DOTALL)  # drop Latin/Italian passages
    text = re.sub(r"<[^>]+>", " ", text)                                         # drop every other tag
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)                                 # re-join hyphenated words
    text = remove_accents(text)
    return " ".join(text.split())                                                # collapse spaces and line breaks

# A quick check on a short sample
print(clean('<head>CARTAS DE SOR MARÍA</head> Pasó por <foreign xml:lang="la">Deo gratias</foreign> este lu-\ngar, señor.'))

## 4. Split every text into passages ("chunks")

An embedding model reads a limited amount of text at a time, and a whole book would be blurred
into a single average. So we cut each text into consecutive passages of ~250 words.
Each passage keeps the `text_id` of its book, so we can later join it with the metadata
(order, century, topic).

In [ ]:
rows = []
for _, row in texts.iterrows():
    with open("corpus/" + row["modern_file"], encoding="utf-8") as f:
        words = clean(f.read()).split()

    # Cut the list of words into slices of CHUNK_WORDS words
    passages = []
    for start in range(0, len(words), CHUNK_WORDS):
        passages.append(words[start:start + CHUNK_WORDS])

    # If the last slice is too short, glue it to the previous one
    if len(passages) > 1 and len(passages[-1]) < MIN_CHUNK_WORDS:
        passages[-2] = passages[-2] + passages[-1]
        passages = passages[:-1]

    for position, passage in enumerate(passages):
        rows.append({
            "text_id": row["text_id"],
            "chunk_n": position,                       # 0 = first passage of the text
            "relative_position": position / len(passages),  # 0 = start of the book, ~1 = end
            "n_words": len(passage),
            "text": " ".join(passage),
        })

chunks = pd.DataFrame(rows)
print(f"{len(chunks):,} passages from {chunks['text_id'].nunique()} texts")
chunks.head()

## 5. Compute the embeddings

We load the model with `sentence-transformers` (already installed in Colab).
`normalize_embeddings=True` makes every vector length 1, so that later the *dot product*
between two passages equals their *cosine similarity*.
Half precision (`float16`) halves memory use and speeds up the GPU with no visible loss.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

print("GPU available:", torch.cuda.is_available())
model = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    model.half()
model.max_seq_length = 512   # ~250 Spanish words are ~350-450 tokens; longer passages get truncated

embeddings = model.encode(
    chunks["text"].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(embeddings.shape)   # (number of passages, 1024)

## 6. Save and download

We store the vectors as `float16` (half the size, ~140 MB) and zip them with the passages table.

In [ ]:
import numpy as np

np.save("embeddings.npy", embeddings.astype("float16"))
chunks.to_csv("chunks.csv.gz", index=False)
!zip -q embeddings.zip embeddings.npy chunks.csv.gz
!ls -lh embeddings.zip

from google.colab import files
files.download("embeddings.zip")